# Descriptive Statistics

Sample coverage, quality-filter accounting, and key-variable distributions for `iv_surface.parquet`. Supports the Data section of the report.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/processed/iv_surface.parquet")
print(f"Total rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

Total rows: 4,445,206
Columns: ['tradingDate', 'ticker', 'split', 'logMoneyness', 'years', 'maturityBucket', 'callPut', 'atmIV', 'realizedVol', 'recentReturn', 'normalizedIV', 'totalVariance', 'midIV', 'expiryDate', 'okey_xx', 'uClose', 'rate', 've']


In [2]:
import textwrap
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG_DIR = Path("../results/figures/descriptive_stats")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def save_table_png(df, filename, title, figsize=None, first_col_label=None, col_widths=None):
    """Render a DataFrame (with its index) as a styled table image, saved under
    results/figures/descriptive_stats/. Mirrors the table style used in
    results/make_summary_table.py so figures are visually consistent.

    Row heights scale with each row's line count (multi-line, textwrap-wrapped
    cells get proportionally taller rows) so wrapped text never overlaps
    adjacent rows. `col_widths` (fractions summing to ~1) lets callers give
    long-text columns more room so wrapped text never floods over the cell's
    left/right border either.
    """
    data = df.reset_index()
    if first_col_label is not None:
        data = data.rename(columns={data.columns[0]: first_col_label})
    col_labels = list(data.columns)
    cell_text = data.astype(str).values.tolist()

    n_rows, n_cols = len(cell_text), len(col_labels)
    row_lines = [1] + [
        max(str(v).count("\n") + 1 for v in row) for row in cell_text
    ]
    total_lines = sum(row_lines)

    if figsize is None:
        figsize = (max(6, n_cols * 1.8), 0.8 + 0.32 * total_lines)

    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")
    ax.set_title(title, fontsize=12, fontweight="bold", pad=14)

    tbl = ax.table(
        cellText=cell_text, colLabels=col_labels, colWidths=col_widths,
        cellLoc="center", loc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)

    row_height = 1.0 / total_lines
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor("#dddddd")
        cell.set_height(row_height * row_lines[row])
        cell.PAD = 0.04
        if row == 0:
            cell.set_facecolor("#2c3e50")
            cell.set_text_props(color="white", weight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#f7f7f7")
        if col == 0:
            cell.set_text_props(ha="left")

    plt.tight_layout()
    out = FIG_DIR / filename
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved to {out}")

## 1. Sample Coverage

Row counts by split × ticker. Held-out tickers (MSFT, JPM, TSLA, XOM) appear only in `test_ticker`.

In [3]:
coverage = (
    df.groupby(["split", "ticker"])
    .size()
    .unstack(fill_value=0)
)
split_order = ["train", "val", "test_temporal", "test_ticker"]
coverage = coverage.reindex(split_order)
coverage["TOTAL"] = coverage.sum(axis=1)
print("Row counts by split × ticker:")
coverage

Row counts by split × ticker:


ticker,AAPL,BA,COST,EOG,GS,JPM,LLY,MSFT,NVDA,SPY,TSLA,XOM,TOTAL
split,,,,,,,,,,,,,
train,175567,150545,239314,89621,166639,0,222717,0,817543,789549,0,0,2651495
val,55992,46779,73086,21269,58021,0,77034,0,210243,256364,0,0,798788
test_temporal,50097,39793,64449,16785,54351,0,71494,0,179288,242210,0,0,718467
test_ticker,0,0,0,0,0,38063,0,81204,0,0,130587,26602,276456


In [4]:
coverage_fmt = coverage.map(lambda v: f"{v:,}")
save_table_png(
    coverage_fmt, "coverage_table.png",
    "Row Counts by Split × Ticker",
    first_col_label="Split",
    figsize=(14, 3.2),
)

Saved to ../results/figures/descriptive_stats/coverage_table.png


/var/folders/5s/fj7mpgnn40b92nyxpk35ysw40000gn/T/ipykernel_68390/2700712496.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Quality Filters Applied in Pipeline

Filters applied in `src/build_dataset.py` before feature computation. See table below (and report Data section) for conditions and rationale.

In [5]:
filter_specs = [
    ("Data quality flag", "error == 0", "Provider flags erroneous quotes"),
    ("Positive IV", "bidIV > 0 and askIV > 0", "Non-positive IV has no Black-Scholes interpretation"),
    ("Minimum maturity", "years >= 7/365 (~7 days)", "Very short-dated options have extreme sensitivity and unreliable IV inversion"),
    ("Positive underlying", "uClose > 0", "Data sanity"),
    ("Open interest", "openInterest > 0", "Excludes stale, illiquid contracts"),
    ("Bid-ask spread", "(askIV - bidIV) / midIV <= 0.50", "Spread > 50% of midIV indicates a wide, unreliable quote"),
    ("Sentinel IV values", "0.001 < midIV < 9.99", "Provider's Black-Scholes solver returns its bound (10.0 ceiling or near-zero floor) instead of a null when inversion fails to converge - 418 rows removed"),
    ("Stock-split masking", "|daily log return| > 40% -> return zeroed before rolling calc", "Unadjusted price series jumps by the split factor (e.g. NVDA's 10-for-1 split); zeroing preserves rolling-window continuity for realizedVol and recentReturn"),
]

filters_df = pd.DataFrame(
    [
        {"Filter": f, "Condition": textwrap.fill(c, width=22), "Rationale": textwrap.fill(r, width=34)}
        for f, c, r in filter_specs
    ]
).set_index("Filter")

save_table_png(
    filters_df, "quality_filters_table.png",
    "Quality Filters Applied (src/build_dataset.py)",
    first_col_label="Filter",
    figsize=(13, 7.5),
    col_widths=[0.18, 0.32, 0.50],
)

Saved to ../results/figures/descriptive_stats/quality_filters_table.png


/var/folders/5s/fj7mpgnn40b92nyxpk35ysw40000gn/T/ipykernel_68390/2700712496.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Descriptive Statistics by Split

Mean/std of surface coordinates, underlying-level features, and targets across splits. See report Data section for variable definitions.

In [6]:
VARS = {
    "logMoneyness":  "Log moneyness",
    "years":         "Years to expiry",
    "atmIV":         "ATM IV",
    "realizedVol":   "Realized vol (30d)",
    "recentReturn":  "Recent return (20d)",
    "normalizedIV":  "Normalized IV  [target]",
    "midIV":         "Mid IV",
    "totalVariance": "Total variance",
}

rows = []
for var, label in VARS.items():
    row = {"Variable": label}
    for split in split_order:
        sub = df[df["split"] == split][var]
        row[split] = f"{sub.mean():.4f} ({sub.std():.4f})"
    rows.append(row)

stats_df = pd.DataFrame(rows).set_index("Variable")
stats_df.index.name = None
stats_df


,train,val,test_temporal,test_ticker
Log moneyness,-0.1318 (0.4185),-0.1455 (0.4182),-0.1585 (0.4373),-0.1330 (0.5225)
Years to expiry,0.6869 (0.6784),0.6576 (0.6334),0.6642 (0.6279),0.7006 (0.6428)
ATM IV,0.3008 (0.1775),0.2992 (0.1431),0.2698 (0.1447),0.3995 (0.2148)
Realized vol (30d),0.3012 (0.1908),0.2973 (0.1868),0.2465 (0.1162),0.4918 (0.2763)
Recent return (20d),0.0432 (0.1031),0.0271 (0.0658),0.0138 (0.0660),0.1399 (0.1563)
Normalized IV [target],1.3232 (0.5965),1.3261 (0.6304),1.4486 (0.7523),1.2464 (0.3739)
Mid IV,0.3572 (0.1889),0.3694 (0.1838),0.3534 (0.1931),0.4749 (0.2681)
Total variance,0.1055 (0.1599),0.1072 (0.1638),0.1044 (0.1790),0.1620 (0.2280)


In [7]:
save_table_png(
    stats_df, "summary_stats_table.png",
    "Descriptive Statistics by Split (mean, std)",
    first_col_label="Variable",
    figsize=(13, 4.5),
)

Saved to ../results/figures/descriptive_stats/summary_stats_table.png


/var/folders/5s/fj7mpgnn40b92nyxpk35ysw40000gn/T/ipykernel_68390/2700712496.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3b. Surface Shape

logMoneyness vs. normalizedIV, colored by maturity bucket (50k-row training sample).

In [8]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Sample from train for readability
rng = np.random.default_rng(42)
sample = df[df["split"] == "train"].sample(n=50_000, random_state=42)

bucket_order = ["7d", "14d", "30d", "60d", "90d", "180d", "1Y"]
colours = plt.cm.plasma(np.linspace(0.1, 0.9, len(bucket_order)))
bucket_colour = dict(zip(bucket_order, colours))

fig, ax = plt.subplots(figsize=(9, 5))

for bucket in bucket_order:
    sub = sample[sample["maturityBucket"] == bucket]
    ax.scatter(
        sub["logMoneyness"], sub["normalizedIV"],
        c=[bucket_colour[bucket]], s=2, alpha=0.25, rasterized=True
    )

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(0, 5)
ax.set_xlabel("Log-moneyness  [log(K/S)]")
ax.set_ylabel("Normalized IV  [midIV / atmIV]")
ax.set_title("IV Surface Shape - Training Set (50k sample)")
ax.axvline(0, color="black", linewidth=0.7, linestyle="--", alpha=0.5)
ax.axhline(1, color="black", linewidth=0.7, linestyle="--", alpha=0.5)

legend_patches = [
    mpatches.Patch(color=bucket_colour[b], label=b) for b in bucket_order
]
ax.legend(handles=legend_patches, title="Maturity", fontsize=8,
          title_fontsize=8, loc="upper center", ncol=7,
          bbox_to_anchor=(0.5, -0.13), frameon=False)

plt.tight_layout()
plt.savefig("../results/figures/descriptive_stats/surface_shape.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to results/figures/descriptive_stats/surface_shape.png")

Saved to results/figures/descriptive_stats/surface_shape.png


/var/folders/5s/fj7mpgnn40b92nyxpk35ysw40000gn/T/ipykernel_68390/1419537535.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. normalizedIV Distribution and Outliers

Tail values (deep OTM, near-zero vega) are retained rather than filtered — see report Data section.

In [9]:
thresholds = [1.5, 2.0, 3.0, 5.0, 10.0]
total = len(df)

rows = []
for t in thresholds:
    n = (df["normalizedIV"] > t).sum()
    rows.append({"normalizedIV >": t, "Count": n, "% of dataset": f"{100*n/total:.3f}%"})

tail_df = pd.DataFrame(rows).set_index("normalizedIV >")
print("normalizedIV tail distribution (full dataset):")
tail_df

normalizedIV tail distribution (full dataset):


,Count,% of dataset
normalizedIV >,,
1.5,990508,22.283%
2.0,437236,9.836%
3.0,125620,2.826%
5.0,16139,0.363%
10.0,0,0.000%


In [10]:
save_table_png(
    tail_df, "normalizediv_tail_table.png",
    "normalizedIV Tail Distribution (full dataset)",
    first_col_label="normalizedIV >",
    figsize=(7, 3.2),
)

Saved to ../results/figures/descriptive_stats/normalizediv_tail_table.png


/var/folders/5s/fj7mpgnn40b92nyxpk35ysw40000gn/T/ipykernel_68390/2700712496.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Distribution of normalizedIV by split
niv_by_split = (
    df.groupby("split")["normalizedIV"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
    .reindex(split_order)
    .round(4)
)
niv_by_split["count"] = niv_by_split["count"].astype(int)
niv_by_split

,count,mean,std,min,25%,50%,75%,95%,99%,max
split,,,,,,,,,,
train,2651495,1.3232,0.5965,0.4127,0.9971,1.1406,1.4258,2.4384,3.9024,8.5655
val,798788,1.3261,0.6304,0.4809,0.9895,1.1324,1.4179,2.4477,4.1877,8.1268
test_temporal,718467,1.4486,0.7523,0.5919,1.0175,1.1959,1.5752,2.8780,4.6996,9.5312
test_ticker,276456,1.2464,0.3739,0.4636,1.0111,1.1331,1.3530,1.9927,2.7133,6.7180


In [12]:
save_table_png(
    niv_by_split, "normalizediv_distribution_table.png",
    "normalizedIV Distribution by Split",
    first_col_label="Split",
    figsize=(13, 3.2),
)

Saved to ../results/figures/descriptive_stats/normalizediv_distribution_table.png


/var/folders/5s/fj7mpgnn40b92nyxpk35ysw40000gn/T/ipykernel_68390/2700712496.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Call/Put and Maturity Bucket Composition

In [13]:
# Call/put split
cp = df.groupby(["split", "callPut"]).size().unstack(fill_value=0).reindex(split_order)
cp.columns = ["Put (0)", "Call (1)"]
cp["Call %"] = (cp["Call (1)"] / (cp["Call (1)"] + cp["Put (0)"]) * 100).round(1)
print("Call/Put composition by split:")
print(cp)

print()

# Maturity bucket composition
bucket_order = ["7d", "14d", "30d", "60d", "90d", "180d", "1Y"]
mb = (
    df.groupby(["split", "maturityBucket"])
    .size()
    .unstack(fill_value=0)
    .reindex(split_order)
    .reindex(columns=bucket_order, fill_value=0)
)
print("Row count by split × maturity bucket:")
mb

Call/Put composition by split:
               Put (0)  Call (1)  Call %
split                                   
train          1333342   1318153    49.7
val             405541    393247    49.2
test_temporal   365765    352702    49.1
test_ticker     134970    141486    51.2



Row count by split × maturity bucket:


maturityBucket,7d,14d,30d,60d,90d,180d,1Y
split,,,,,,,
train,150904,115856,347775,199895,340341,434844,1061880
val,46842,35337,106576,60675,113310,110095,325953
test_temporal,41871,31445,96463,57191,90630,104939,295928
test_ticker,13473,13105,37760,15831,31074,45373,119840


In [14]:
cp_fmt = cp.copy()
cp_fmt["Put (0)"] = cp_fmt["Put (0)"].map(lambda v: f"{v:,}")
cp_fmt["Call (1)"] = cp_fmt["Call (1)"].map(lambda v: f"{v:,}")
cp_fmt["Call %"] = cp_fmt["Call %"].map(lambda v: f"{v:.1f}%")

save_table_png(
    cp_fmt, "callput_composition_table.png",
    "Call / Put Composition by Split",
    first_col_label="Split",
    figsize=(8, 3),
)

mb_fmt = mb.map(lambda v: f"{v:,}")
save_table_png(
    mb_fmt, "maturity_bucket_table.png",
    "Row Count by Split × Maturity Bucket",
    first_col_label="Split",
    figsize=(13, 3),
)

Saved to ../results/figures/descriptive_stats/callput_composition_table.png
Saved to ../results/figures/descriptive_stats/maturity_bucket_table.png


/var/folders/5s/fj7mpgnn40b92nyxpk35ysw40000gn/T/ipykernel_68390/2700712496.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/5s/fj7mpgnn40b92nyxpk35ysw40000gn/T/ipykernel_68390/2700712496.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Date Range and Trading Day Coverage

In [15]:
date_range = df.groupby("split")["tradingDate"].agg(["min", "max", "nunique"]).reindex(split_order)
date_range.columns = ["First date", "Last date", "Unique trading days"]
date_range

,First date,Last date,Unique trading days
split,,,
train,2024-01-31,2024-08-30,148
val,2024-09-03,2024-10-31,43
test_temporal,2024-11-01,2024-12-31,41
test_ticker,2024-11-01,2024-12-31,41


In [16]:
save_table_png(
    date_range, "date_range_table.png",
    "Date Range and Trading-Day Coverage by Split",
    first_col_label="Split",
    figsize=(9, 3),
)

Saved to ../results/figures/descriptive_stats/date_range_table.png


/var/folders/5s/fj7mpgnn40b92nyxpk35ysw40000gn/T/ipykernel_68390/2700712496.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
